In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_25.pickle

In [ ]:
%%cudf.pandas.profile
### cell 25 ###
# Filter for USA and India (GPU)
us_ind = df[(df['first_country'] == 'USA') | (df['first_country'] == 'India')]

# Create indicator columns for each country (GPU)
us_ind['USA'] = (us_ind['first_country'] == 'USA').astype('int32')
us_ind['India'] = (us_ind['first_country'] == 'India').astype('int32')

# Aggregate yearly counts per country (GPU)
yearly_counts = us_ind.groupby('year_added')[['USA','India']].sum()

# Compute half of the total counts per year for centering (GPU)
half_total = (yearly_counts['USA'] + yearly_counts['India']) / 2

# Build data_sub with baseline and lobe positions (GPU)
data_sub = yearly_counts.copy()
data_sub.insert(0, 'base', -half_total)
data_sub['USA'] = yearly_counts['USA'] - half_total
data_sub['India'] = half_total

# (Optional) Ensure years are sorted
# data_sub = data_sub.sort_index()